#### 12.1
European Options GBSM including Greeks, using test12_1.csv and testout_12.1.csv

In [129]:
import numpy as np
import pandas as pd
import math
from scipy.stats import norm

In [130]:
data = pd.read_csv('../testfiles/data/test12_1.csv').dropna()
data

,ID,Option Type,Underlying,Strike,DaysToMaturity,DayPerYear,RiskFreeRate,DividendRate,ImpliedVol
0,1.0,Call,100.0,100.0,50.0,365.0,0.045,0.00,0.2
1,2.0,Put,100.0,100.0,50.0,365.0,0.045,0.00,0.2
2,3.0,Call,100.0,90.0,400.0,365.0,0.045,0.02,0.4
3,4.0,Put,100.0,110.0,400.0,365.0,0.045,0.02,0.4


In [131]:
output = pd.DataFrame()
output['ID'] = data['ID']

for i in range(data.shape[0]):
    S = data.loc[i, 'Underlying']
    K = data.loc[i, 'Strike']
    T = data.loc[i, 'DaysToMaturity'] / data.loc[i, 'DayPerYear']
    r = data.loc[i, 'RiskFreeRate']
    q = data.loc[i, 'DividendRate']
    sigma = data.loc[i, 'ImpliedVol']
    
    d1 = (math.log(S/K) + (r - q + 1/2 * sigma**2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)

    helper = math.exp(-q * T) * norm.cdf(d1)

    output.loc[i, 'Gamma'] = math.exp(-q * T) * norm.pdf(d1) / (S * sigma * math.sqrt(T))
    output.loc[i, 'Vega'] = S * math.exp(-q * T) * norm.pdf(d1) * math.sqrt(T)
    
    if data.loc[i, 'Option Type'] == "Call":
        output.loc[i, 'Value'] = S * helper - K * math.exp(-r * T) * norm.cdf(d2)
        output.loc[i, 'Delta'] = helper
        output.loc[i, 'Rho'] = K * T * math.exp(-r * T) * norm.cdf(d2)
        output.loc[i, 'Theta'] = -S * math.exp(-q * T) * norm.pdf(d1) * sigma / (2 * math.sqrt(T)) - r * K * math.exp(-r * T) * norm.cdf(d2) + q * S * helper
                                                                                              
    else:
        output.loc[i, 'Value'] = K * math.exp(-r * T) * norm.cdf(-d2) - S * math.exp(-q * T) * norm.cdf(-d1)
        output.loc[i, 'Delta'] = math.exp(-q * T) * (norm.cdf(d1) - 1)
        output.loc[i, 'Rho'] = -K * T * math.exp(-r * T) * norm.cdf(-d2)
        output.loc[i, 'Theta'] = -S * math.exp(-q * T) * norm.pdf(d1) * sigma / (2 * math.sqrt(T)) + r * K * math.exp(-r * T) * norm.cdf(-d2) - q * S * math.exp(-q * T) * norm.cdf(-d1)

In [132]:
new_order = ['ID', 'Value', 'Delta', 'Gamma', 'Vega', 'Rho', 'Theta']
output = output[new_order]
output

,ID,Value,Delta,Gamma,Vega,Rho,Theta
0,1.0,3.260824,0.547872,0.053506,14.659079,7.058414,-13.019817
1,2.0,2.646281,-0.452128,0.053506,14.659079,-6.556032,-8.547471
2,3.0,22.043329,0.685508,0.008115,35.571438,50.967099,-7.213608
3,4.0,20.449083,-0.470751,0.009310,40.812329,-73.999110,-5.351164


#### 12.2
American Options with continuous Dividends including Greeks，using test12_1.csv and testout_12.2.csv

In [133]:
def bt_american(call, underlying, strike, ttm, rf, b, ivol, N):
    dt = ttm / N
    u = np.exp(ivol * np.sqrt(dt))
    d = 1 / u
    pu = (np.exp(b * dt) - d) / (u - d)
    pd = 1.0 - pu
    df = np.exp(-rf * dt)
    z = 1 if call else -1

    # price tree at maturity
    prices = np.array([underlying * (u ** (N - i)) * (d ** i) for i in range(N + 1)])

    # Initialize option values at maturity
    values = np.maximum(z * (prices - strike), 0)

    # Backward induction
    for step in range(N - 1, -1, -1):
        for i in range(step + 1):
            price = underlying * (u ** (step - i)) * (d ** i)
            # Continuation value
            values[i] = df * (pu * values[i] + pd * values[i + 1])
            # Early exercise value
            exercise = max(0, z * (price - strike))
            values[i] = max(values[i], exercise)

    return values[0]

In [134]:
def american_option_greeks(S, K, T, r, q, sigma, option_type):
    call = (option_type == 'Call')
    b = r - q
    N = 500

    price = bt_american(call, S, K, T, r, b, sigma, N)

    dS = 1.0
    price_up = bt_american(call, S + dS, K, T, r, b, sigma, N)
    price_down = bt_american(call, S - dS, K, T, r, b, sigma, N)
    delta = (price_up - price_down) / (2 * dS)

    gamma = (price_up - 2 * price + price_down) / (dS ** 2)

    dv = 0.01
    price_vol_up = bt_american(call, S, K, T, r, b, sigma + dv, N)
    vega = (price_vol_up - price) / dv

    dr = 0.001
    price_r_up = bt_american(call, S, K, T, r + dr, b, sigma, N)
    rho = (price_r_up - price) / dr

    dt = 1 / 365
    price_t_up = bt_american(call, S, K, T + dt, r, b, sigma, N)
    theta = (price_t_up - price) / dt

    return price, delta, gamma, vega, rho, theta

In [135]:
results2 = []

for i in range(data.shape[0]):
    S = data.loc[i, 'Underlying']
    K = data.loc[i, 'Strike']
    T = data.loc[i, 'DaysToMaturity'] / data.loc[i, 'DayPerYear']
    r = data.loc[i, 'RiskFreeRate']
    q = data.loc[i, 'DividendRate']
    sigma = data.loc[i, 'ImpliedVol']
    option_type = data.loc[i, 'Option Type']

    value, delta, gamma, vega, rho, theta = american_option_greeks(S, K, T, r, q, sigma, option_type)
    results2.append([P, delta, gamma, vega, rho, theta])

output2 = pd.DataFrame()
output2['ID'] = data['ID']
output2[['Value', 'Delta', 'Gamma', 'Vega', 'Rho', 'Theta']] = results2

output2

,ID,Value,Delta,Gamma,Vega,Rho,Theta
0,1.0,21.019233,0.547643,0.059244,14.653661,-0.446455,12.960975
1,2.0,21.019233,-0.463483,0.060507,14.615371,-0.253688,8.887079
2,3.0,21.019233,0.680852,0.008986,35.904557,-22.792528,7.275168
3,4.0,21.019233,-0.490073,0.002592,40.744104,-14.083944,5.951540


#### 12.3
American Options with discreet Dividends, using test12_3.csv and testout_12.3.csv

In [136]:
data2 = pd.read_csv('../testfiles/data/test12_3.csv')
data2

,ID,Option Type,Underlying,Strike,DaysToMaturity,DayPerYear,RiskFreeRate,ImpliedVol,DividendDates,DividendAmts
0,1,Call,100,100,250,365,0.045,0.4,"75,150",".01,.01"
1,2,Put,100,100,250,365,0.045,0.4,"75,150",".01,.01"


In [137]:
def american_option_discrete_div(S, K, T, r, sigma, option_type, div_dates, div_amts, steps):
    dt = T / steps
    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u
    p = (np.exp(r * dt) - d) / (u - d)
    discount = np.exp(-r * dt)

    # Convert dividend dates to step indices
    div_steps = [int(div_date * steps) for div_date in div_dates]

    # Build tree
    asset_tree = np.zeros((steps + 1, steps + 1))
    asset_tree[0, 0] = S

    for i in range(1, steps + 1):
        # Check if dividend is paid at this step
        div_paid = sum(div_amts[j] for j, ds in enumerate(div_steps) if ds == i)

        for j in range(i + 1):
            if j == 0:
                asset_tree[i, j] = asset_tree[i-1, j] * u - div_paid
            elif j == i:
                asset_tree[i, j] = asset_tree[i-1, j-1] * d - div_paid
            else:
                asset_tree[i, j] = asset_tree[i-1, j-1] * d - div_paid

    # Initialize option values at maturity
    option_values = np.zeros(steps + 1)
    for i in range(steps + 1):
        ST = S
        for j in range(steps):
            if j < i:
                ST *= u
            else:
                ST *= d
            # Subtract dividends paid before maturity
            for k, ds in enumerate(div_steps):
                if ds <= j + 1:
                    ST -= div_amts[k]

        if option_type == 'Call':
            option_values[i] = max(ST - K, 0)
        else:
            option_values[i] = max(K - ST, 0)

    # Recalc proper tree structure
    ST_values = np.zeros(steps + 1)
    for i in range(steps + 1):
        ST_values[i] = S * (u ** (steps - i)) * (d ** i)
        for k, ds in enumerate(div_steps):
            if ds <= steps:
                ST_values[i] -= div_amts[k] * (u ** max(0, steps - ds - i)) * (d ** min(i, steps - ds))

    if option_type == 'Call':
        option_values = np.maximum(ST_values - K, 0)
    else:
        option_values = np.maximum(K - ST_values, 0)

    # Backward induction
    for j in range(steps - 1, -1, -1):
        for i in range(j + 1):
            S_node = S * (u ** (j - i)) * (d ** i)

            # Subtract dividends already paid
            for k, ds in enumerate(div_steps):
                if ds <= j:
                    S_node -= div_amts[k]

            option_values[i] = discount * (p * option_values[i] + (1 - p) * option_values[i + 1])

            # Early exercise
            if option_type == 'Call':
                exercise_value = max(S_node - K, 0)
            else:
                exercise_value = max(K - S_node, 0)

            option_values[i] = max(option_values[i], exercise_value)

    return option_values[0]

In [138]:
results3 = []

for _, row in data2.iterrows():
    
    S = row['Underlying']
    K = row['Strike']
    T = row['DaysToMaturity'] / row['DayPerYear']
    r = row['RiskFreeRate']
    sigma = row['ImpliedVol']
    option_type = row['Option Type']

    # Parse dividend dates and amounts
    div_dates_str = str(row['DividendDates']).split(',')
    div_amts_str = str(row['DividendAmts']).split(',')

    div_dates = [float(d.strip()) / row['DayPerYear'] for d in div_dates_str]
    div_amts = [float(a.strip()) for a in div_amts_str]

    value, delta, gamma, vega, rho, theta = american_option_greeks_discrete(S, K, T, r, sigma, option_type, div_dates, div_amts)

    results3.append({
        'ID': int(row['ID']),
        'Value': value
    })

output3 = pd.DataFrame(results3)
output3

,ID,Value
0,1,14.504598
1,2,11.777884
